# Point 1 — Study and analysis of the RBM on MNIST (digits 0, 1, 2)

This notebook trains a Restricted Boltzmann Machine on digits **0 / 1 / 2** of MNIST
and analyses its behaviour following Hinton's *Practical Guide* (2010):

1. training and training metrics;
2. Contrastive Divergence update analysis + **gradient RMS** monitoring;
3. visualisation of the learned weights (*receptive fields*);
4. generative behaviour via Gibbs chains;
5. effect of the number of hidden units *L*.

> Cells must be executed **in order, top to bottom**. The notebook is self-contained.


## 0. Setup and imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

from rbm import RBM, classify_nearest, compute_prototypes, load_results
from rbm_plots import (
    plot_metrics, plot_filters, plot_cd_analysis,
    plot_weight_evolution, plot_chain_digits, plot_convergence_comparison,
)

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 12
})


## 1. Data loading and preprocessing

We select digits **0, 1, 2**: the more diverse the digits in the set, the harder the
learning task (as stated in the assignment).

Preprocessing normalises pixels to `[0, 1]` and adds small Gaussian noise (σ = 0.01)
as regularisation.

> The data remain **continuous** in `[0, 1]` and are not binarised: the RBM interprets them
> as activation probabilities of the visible units.


In [ ]:
X_original, Y_original = fetch_openml("mnist_784", version=1,
                                      return_X_y=True, as_frame=False)

In [ ]:
# --- DATA LOADING ---
X_original = np.array(X_original, dtype=np.float32)
Y_original = np.array(Y_original, dtype=int)

mask = (Y_original == 0) | (Y_original == 1) | (Y_original == 2)
X_filtered = X_original[mask]
Y_filtered = Y_original[mask]

print(f"Full dataset  : {X_original.shape}")
print(f"Filtered 0,1,2: {X_filtered.shape}")


In [ ]:
# --- PREPROCESSING ---
def preprocess_mnist(images):
    """Normalise to [0,1] and add small Gaussian noise (regularisation)."""
    images = images / 255.0
    images = images + np.random.normal(0, 0.01, images.shape)
    return np.clip(images, 0, 1)

X_train = preprocess_mnist(X_filtered)
D = X_train.shape[1]          # number of visible units (784)

print(f"Data range    : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Dataset mean  : {X_train.mean():.3f}")
print(f"D (visible)   : {D}")


## 2. Base model — training

Main hyperparameters: `L = 12` hidden units, **RMSprop** optimiser, constant learning rate
0.05, **L1** regularisation (γ = 0.001), **CD-2**.

`hinton_init=True` initialises the visible biases to `log[p_i / (1 − p_i)]` where `p_i`
is the fraction of training images in which pixel *i* is active (Hinton, Sec. 8).

> **Weight initialisation:** the notebook uses a Glorot-like scale `sqrt(4/(D+L))` (≈ 0.07),
> larger than the **0.01** recommended by Hinton (Sec. 8). Larger values speed up early
> learning but may slightly worsen the final model.


In [ ]:
glorot_scale = np.sqrt(4.0 / (D + 12))

pro_model = RBM(
    n_v=D,
    n_h=12,                    # L = hidden units
    seed=12345,
    w_init_scale=glorot_scale,
    optimizer="rmsprop",
    lr=0.05,
    lr_schedule="constant",
    gamma=0.001,               # L1 weight penalty
    weight_decay=0.0,          # L2 disabled
    potts=False,
    spins=False,
)

pro_results = pro_model.train(
    X_train,
    Nepoch=150,
    Nmini=20,
    N_ini=10,
    N_fin=500,
    Nt=2,                      # CD-k steps
    persistent=False,          # standard CD (not PCD)
    hinton_init=True,
    verbose=True,
)


## 3. Training metrics

The four panels show: free energy on data vs. *fantasy* particles, their gap,
reconstruction error (MSE), and pseudo-likelihood.

> Reconstruction error should be **monitored but not trusted absolutely** (Hinton, Sec. 5):
> it is not the objective optimised by CD and can decrease simply because the Gibbs chain
> mixes more slowly.


In [ ]:
plot_metrics(pro_results)

## 4. Contrastive Divergence analysis

Decomposition of the gradient into the **positive** ⟨v·h⟩_data and **negative** ⟨v·h⟩_model
phases, and monitoring of the **gradient RMS** during training. At convergence the two phases
align and the gradient RMS decreases.


In [ ]:
plot_cd_analysis(pro_results)

## 5. Learned filters

Each column of the weight matrix W is a *receptive field*, visualised as a 28×28 image
with a shared colour scale (Hinton, Sec. 15).


In [ ]:
plot_filters(pro_results["weights"]["w"], side=28)

## 6. Weight evolution

Filters and visible bias at selected epochs, showing how features form over training.


In [ ]:
plot_weight_evolution(pro_results, side=28, checkpoints=[1, 25, 50, 100, 150])

## 7. Generative behaviour — Gibbs chains

Starting from a real image we run a Gibbs chain for 100 steps and classify each state
using the nearest **prototype** (per-class mean image).

This illustrates **quasi-ergodicity** of CD: the chain tends to stay on the same digit
and rarely transitions between hidden representations — motivating the energy-barrier
analysis in Point 3.


In [ ]:
# Class prototypes (per-class mean image)
prototypes = compute_prototypes(X_train, Y_filtered)

# Run 100-step Gibbs chains starting from 3 real images
chains = [pro_model.sample_chain(X_train[i], n_steps=100) for i in range(3)]

# Classify each chain state using the nearest prototype
sequences = [[classify_nearest(chain[t], prototypes) for t in range(100)]
             for chain in chains]

plot_chain_digits(sequences, start_labels=[0, 1, 2], n_classes=3)


## 8. Effect of the parameter L (number of hidden units)

We study the effect of L ∈ {1, 2, 4, 8, 12, 16, 20, 24} on reconstruction error,
pseudo-likelihood, gradient RMS and free-energy gap. Models are saved to `L_run/<L>`
and reloaded for comparison.


In [ ]:
L_list = [1, 2, 4, 8, 12, 16, 20, 24]

for l in L_list:
    print(f"Training with L = {l} ...")

    glorot_scale_l = np.sqrt(4.0 / (D + l))

    L_model = RBM(
        n_v=D,
        n_h=l,
        seed=42,
        w_init_scale=glorot_scale_l,
        optimizer="rmsprop",
        lr=0.05,
        lr_schedule="constant",
        gamma=0.001,
        weight_decay=0.0,
        potts=False,
        spins=False,
    )

    L_model.train(
        X_train,
        Nepoch=150, Nmini=20, N_ini=10, N_fin=500, Nt=2,
        persistent=False,
        hinton_init=True,
        save_dir=f"L_run/{l}",
        verbose=False,
    )

print("All L models trained.")


### 8.1 Comparison across L values

We load the saved results once into the `runs` dictionary and reuse it for all
comparison plots.


In [ ]:
runs = {f"L={l}": load_results(f"L_run/{l}") for l in L_list}


In [ ]:
# Summary: final reconstruction error and pseudo-likelihood vs L
RE_L = [np.mean(runs[f"L={l}"]["metrics"]["reconstruction_error"][-100:]) for l in L_list]
PL_L = [np.mean(runs[f"L={l}"]["metrics"]["pseudo_likelihood"][-50:])    for l in L_list]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(L_list, RE_L, marker="o"); ax[0].set(title="Reconstruction Error vs L",
                                                xlabel="L", ylabel="MSE")
ax[1].plot(L_list, PL_L, marker="o"); ax[1].set(title="Pseudo-Likelihood vs L",
                                                xlabel="L", ylabel="log PL")
plt.tight_layout(); plt.show()


In [ ]:
# Convergence curves across L values
plot_convergence_comparison(runs, metric="reconstruction_error",
                            title="Reconstruction Error vs L")


In [ ]:
plot_convergence_comparison(runs, metric="gradient_rms",
                            title="Gradient RMS vs L")


In [ ]:
plot_convergence_comparison(runs, metric="free_energy_gap",
                            title="Free-energy gap vs L")
